In [1]:

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


# DATA Check

In [2]:

import pandas as pd

df = pd.read_excel('/content/drive/MyDrive/Thesis/version30k.xlsx')
print(df.info())
print(df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30117 entries, 0 to 30116
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Abuse Text  30117 non-null  object
 1   Aspects     30117 non-null  object
 2   Outcome     30117 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 706.0+ KB
None
                                          Abuse Text  Aspects  Outcome
0  এই গুলার জায়গামতো চুঁচড়া লাগিয়ে দেয়া দরকার! যত...    Toxic        1
1  এই গুলার জায়গামতো চুঁচড়া লাগিয়ে দেয়া দরকার! যত...  Obscene        0
2  এই গুলার জায়গামতো চুঁচড়া লাগিয়ে দেয়া দরকার! যত...   Insult        1
3   এই মাদ্রাসার হোগুরকে ডিলডো মেরে শাস্তি দেয়া হোক।    Toxic        1
4   এই মাদ্রাসার হোগুরকে ডিলডো মেরে শাস্তি দেয়া হোক।  Obscene        0


In [3]:
max_char_count = df['Abuse Text'].astype(str).apply(len).max()
min_char_count = df['Abuse Text'].astype(str).apply(len).min()

max_word_count = df['Abuse Text'].astype(str).apply(lambda x: len(x.split())).max()
min_word_count = df['Abuse Text'].astype(str).apply(lambda x: len(x.split())).min()

print("Maximum Character Count:", max_char_count)
print("Minimum Character Count:", min_char_count)
print("Maximum Word Count:", max_word_count)
print("Minimum Word Count:", min_word_count)

Maximum Character Count: 143
Minimum Character Count: 1
Maximum Word Count: 20
Minimum Word Count: 1


# Data Process

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [5]:
# Download NLTK data
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

**Cleaning the Text:**
Converts text to lowercase.
Removes special characters, digits, and extra whitespaces for cleaner data.

**Tokenization and Lemmatization:**
Tokenizes the text into individual words.
Removes stopwords (e.g., "the", "and") to reduce noise in the data.
Uses lemmatization to convert words to their base form (e.g., "running" → "run").

**Encoding Categorical Features:**
The Aspects column is encoded into numerical labels using LabelEncoder.

**Numerical Feature Extraction:**
The Abuse Text is transformed into numerical features using TfidfVectorizer, which converts text into a sparse matrix of TF-IDF scores.

**Combining Features:**
Combines the numerical text features with the encoded Aspects column to form the final input feature set.

**Dataset Splitting:**
Splits the dataset into training and testing sets using an 80-20 split while preserving the distribution of the target variable (stratify=y).




# Handle Missing Data

In [6]:
import pandas as pd

def handle_missing_data(df, required_columns):
    if not all(col in df.columns for col in required_columns):
        raise ValueError(f"The dataset must contain the following columns: {required_columns}")
    return df.dropna(subset=required_columns)

# Clean Text Data

In [7]:
import re

def clean_text_column(df, text_column):
    def clean_text(text):
        text = str(text).strip()  # Convert to string before stripping
        text = re.sub(r'[^\u0980-\u09FF\s]', ' ', text)  # Keep only Bengali characters and spaces
        text = re.sub(r'\s+', ' ', text)  # Remove extra spaces
        return text.strip()

    df['Cleaned_Text'] = df[text_column].apply(clean_text)
    return df

# Tokenization and Stopword Removal

In [8]:
from nltk.corpus import stopwords
import nltk

nltk.download('stopwords')

def tokenize_and_remove_stopwords(df, text_column):

    stop_words = set(stopwords.words('bengali'))  # Define Bengali stopwords

    def tokenize_and_remove(text):
        tokens = text.split()  # Tokenize by splitting on spaces
        tokens = [word for word in tokens if word not in stop_words]  # Remove stopwords
        return ' '.join(tokens)  # Join tokens back into a single string

    df['Processed_Text'] = df[text_column].apply(tokenize_and_remove)
    return df

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


# Encode the Aspects Column

In [9]:
from sklearn.preprocessing import LabelEncoder

def encode_aspects_column(df, aspects_column):
    label_encoder = LabelEncoder()
    df['Encoded_Aspects'] = label_encoder.fit_transform(df[aspects_column])
    return df, label_encoder

# Convert Text to Numerical Features

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

def vectorize_text(df, text_column, max_features=5000):
    tfidf_vectorizer = TfidfVectorizer(max_features=max_features)
    X_text_tfidf = tfidf_vectorizer.fit_transform(df[text_column]).toarray()
    return X_text_tfidf, tfidf_vectorizer

# Combine Features and Split Data

In [11]:
from sklearn.model_selection import train_test_split
import numpy as np

def prepare_features_and_split(X_text_tfidf, X_aspects, y, test_size=0.2, random_state=42):
    # Combine text features and aspect features
    X_aspects = X_aspects.reshape(-1, 1)  # Reshape for concatenation
    X = np.hstack((X_text_tfidf, X_aspects))  # Combine features

    # Split the dataset
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state, stratify=y)
    return X_train, X_test, y_train, y_test

# Final Preprocessing Function

In [12]:
def preprocess_data(df):

    required_columns = ['Abuse Text', 'Aspects', 'Outcome']

    # Step 1: Handle missing values
    df = handle_missing_data(df, required_columns)

    # Step 2: Clean the 'Abuse Text' column
    df = clean_text_column(df, 'Abuse Text')

    # Step 3: Tokenize and remove stopwords
    df = tokenize_and_remove_stopwords(df, 'Cleaned_Text')

    # Step 4: Encode the 'Aspects' column
    df, label_encoder = encode_aspects_column(df, 'Aspects')

    # Step 5: Vectorize the text data
    X_text_tfidf, tfidf_vectorizer = vectorize_text(df, 'Processed_Text')

    # Step 6: Prepare features and split the dataset
    X_train, X_test, y_train, y_test = prepare_features_and_split(
        X_text_tfidf, df['Encoded_Aspects'].values, df['Outcome'].values
    )

    return X_train, X_test, y_train, y_test, tfidf_vectorizer, label_encoder

In [13]:
X_train, X_test, y_train, y_test, tfidf_vectorizer, label_encoder = preprocess_data(df)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (24093, 2327)
X_test shape: (6024, 2327)
y_train shape: (24093,)
y_test shape: (6024,)


**Handling Missing Data:**
Ensures the dataset contains the required columns: Abuse Text, Aspects, and Outcome.
Drops rows with missing values to avoid errors during processing.

**Cleaning the Text:**
Normalizes the Abuse Text column by:
Retaining only Bengali characters.
Removing special characters, punctuation, and digits.
Reducing multiple spaces into a single space.

**Tokenization and Stopword Removal:**
Splits the cleaned text into individual words (tokenization).
Removes Bengali stopwords (e.g., common words like "এবং", "এই") using NLTK's Bengali stopwords list to reduce noise in the data.

**Encoding Categorical Features:**
Converts the Aspects column into numerical labels using LabelEncoder.

**Numerical Feature Extraction:**
Transforms the processed Abuse Text into numerical features using TfidfVectorizer, which computes Term Frequency-Inverse Document Frequency (TF-IDF) scores for a maximum of 5000 features.

**Combining Features:**
Combines the TF-IDF text features with the encoded Aspects column to create a final feature set suitable for machine learning models.

**Dataset Splitting:**
Splits the combined feature set (X) and the target variable (Outcome) into training and testing sets.
Uses an 80-20 split while preserving the class distribution of the target variable (stratify=y).

# BERT

# Adjust Preprocessing for BERT



In [14]:
def preprocess_data_for_bert(df):

    required_columns = ['Abuse Text', 'Aspects', 'Outcome']

    # Step 1: Handle missing values
    df = handle_missing_data(df, required_columns)

    # Step 2: Clean the 'Abuse Text' column
    df = clean_text_column(df, 'Abuse Text')

    # Step 3: Tokenize and remove stopwords
    df = tokenize_and_remove_stopwords(df, 'Cleaned_Text')

    # Step 4: Encode the 'Outcome' column (target labels)
    y = df['Outcome'].values  # Labels

    # Step 5: Split the dataset
    train_texts, test_texts, y_train, y_test = train_test_split(
        df['Processed_Text'], y, test_size=0.2, random_state=42, stratify=y
    )

    return train_texts.tolist(), test_texts.tolist(), y_train, y_test

In [15]:
train_texts, test_texts, y_train, y_test = preprocess_data_for_bert(df)

print(f"Number of training samples: {len(train_texts)}")
print(f"Number of testing samples: {len(test_texts)}")

Number of training samples: 24093
Number of testing samples: 6024


In [16]:
pip install transformers torch

# Load BERT Tokenizer

In [17]:
from transformers import BertTokenizer

# Load BERT tokenizer
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

# Prepare the Dataset

In [18]:
import torch
from torch.utils.data import Dataset

class BengaliDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,  # Add [CLS] and [SEP]
            max_length=self.max_length,
            truncation=True,
            padding='max_length',
            return_tensors='pt'  # Return PyTorch tensors
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),  # Remove batch dimension
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.long)
        }

# Create DataLoaders

In [19]:
from torch.utils.data import DataLoader

# Define maximum sequence length and batch size
max_length = 128
batch_size = 16

# Create datasets
train_dataset = BengaliDataset(train_texts, y_train, bert_tokenizer, max_length)
test_dataset = BengaliDataset(test_texts, y_test, bert_tokenizer, max_length)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Load Pre-Trained BERT Model

In [25]:
from transformers import BertForSequenceClassification

# Load pre-trained BERT model with a classification head
bert_model = BertForSequenceClassification.from_pretrained(
    'bert-base-multilingual-cased',
    num_labels=2  # Binary classification
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# Define Optimizer and Loss Function

In [26]:
import torch.optim as optim
import torch.nn as nn

# Define optimizer and loss function
optimizer = optim.AdamW(bert_model.parameters(), lr=2e-5)  # Learning rate
loss_fn = nn.CrossEntropyLoss()

# Train the BERT Model

In [27]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
bert_model.to(device)  # Move model to GPU

epochs = 1  # Number of epochs

for epoch in range(epochs):
    bert_model.train()  # Set model to training mode
    total_loss = 0
    correct_preds = 0
    total_samples = 0

    for batch in train_loader:
        # Move batch data to device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        # Forward pass
        outputs = bert_model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        logits = outputs.logits

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Track loss and accuracy
        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        correct_preds += (preds == labels).sum().item()
        total_samples += labels.size(0)

    # Print epoch statistics
    train_accuracy = correct_preds / total_samples
    print(f"Epoch {epoch + 1}/{epochs}, Loss: {total_loss:.4f}, Accuracy: {train_accuracy:.4f}")

Epoch 1/1, Loss: 353.9144, Accuracy: 0.9107


In [23]:
# prompt: genarate roc curve and save the fig fig will be colorful

import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

y_pred_prob = bert_model(torch.tensor(X_test).to(device)).logits[:,1]

fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc="lower right")
plt.savefig('Bertroc_curve.png') # Save the figure
plt.show()

We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.


RuntimeError: The expanded size of the tensor (2327) must match the existing size (512) at non-singleton dimension 1.  Target sizes: [6024, 2327].  Tensor sizes: [1, 512]

# Evaluate the BERT Model

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

bert_model.eval()  # Set model to evaluation mode
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        # Move data to device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        # Forward pass
        outputs = bert_model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        # Store predictions and labels
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

# Print evaluation results
accuracy = accuracy_score(all_labels, all_preds)
print(f"Test Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(all_labels, all_preds))

In [ ]:


from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, matthews_corrcoef


# Print evaluation results
accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds)
recall = recall_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)
mcc = matthews_corrcoef(all_labels, all_preds)

tn, fp, fn, tp = confusion_matrix(all_labels, all_preds).ravel()
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall (Sensitivity): {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Sensitivity: {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"Matthews Correlation Coefficient: {mcc:.4f}")

Matthews Correlation Coefficient (MCC):
Balanced measure of the quality of binary classifications.
Formula:
MCC
=
(
TP
⋅
TN
)
−
(
FP
⋅
FN
)
(
TP
+
FP
)
(
TP
+
FN
)
(
TN
+
FP
)
(
TN
+
FN
)
MCC=
(TP+FP)(TP+FN)(TN+FP)(TN+FN)
​

(TP⋅TN)−(FP⋅FN)
​



In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt


cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="YlGnBu",
            xticklabels=['Predicted 0', 'Predicted 1'],
            yticklabels=['Actual 0', 'Actual 1'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.savefig('Bertconfusion_matrix.png')
plt.show()

# ML Models

# LR

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Train Logistic Regression Model
logistic_model = LogisticRegression(max_iter=200)
logistic_model.fit(X_train, y_train)

# Make Predictions
y_pred_train = logistic_model.predict(X_train)
y_pred_test = logistic_model.predict(X_test)

# Evaluate Model
train_accuracy = accuracy_score(y_train, y_pred_train)
test_accuracy = accuracy_score(y_test, y_pred_test)

print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Testing Accuracy: {test_accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_test))


# RF

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Train Random Forest Model
rf_model = RandomForestClassifier(n_estimators=300, random_state=39)
rf_model.fit(X_train, y_train)

# Make Predictions
y_pred_train = rf_model.predict(X_train)
y_pred_test = rf_model.predict(X_test)

# Evaluate Model
train_accuracy = accuracy_score(y_train, y_pred_train)
test_accuracy = accuracy_score(y_test, y_pred_test)

print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Testing Accuracy: {test_accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_test))

# XGBoost

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

# Train XGBoost Model
xgb_model = XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=6, random_state=42)
xgb_model.fit(X_train, y_train)

# Make Predictions
y_pred_train = xgb_model.predict(X_train)
y_pred_test = xgb_model.predict(X_test)

# Evaluate Model
train_accuracy = accuracy_score(y_train, y_pred_train)
test_accuracy = accuracy_score(y_test, y_pred_test)

print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Testing Accuracy: {test_accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_test))

# LightGBM

In [ ]:
from lightgbm import LGBMClassifier

# Train LightGBM Model
lgbm_model = LGBMClassifier(n_estimators=200, learning_rate=0.1, max_depth=6, random_state=42)
lgbm_model.fit(X_train, y_train)

# Make Predictions
y_pred_train = lgbm_model.predict(X_train)
y_pred_test = lgbm_model.predict(X_test)

# Evaluate Model
train_accuracy = accuracy_score(y_train, y_pred_train)
test_accuracy = accuracy_score(y_test, y_pred_test)

print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Testing Accuracy: {test_accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_test))

# Neural Networks (MLPClassifier)

In [ ]:
from sklearn.neural_network import MLPClassifier

# Train MLP Model
mlp_model = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=300, random_state=41)
mlp_model.fit(X_train, y_train)

# Make Predictions
y_pred_train = mlp_model.predict(X_train)
y_pred_test = mlp_model.predict(X_test)

# Evaluate Model
train_accuracy = accuracy_score(y_train, y_pred_train)
test_accuracy = accuracy_score(y_test, y_pred_test)

print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Testing Accuracy: {test_accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_test))

# Ensemble Voting Classifier

In [ ]:
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import RandomForestClassifier  # Import RandomForestClassifier
from xgboost import XGBClassifier  # Import XGBClassifier
from sklearn.svm import SVC  # Import SVC

# Create Individual Models
rf = RandomForestClassifier(n_estimators=100, random_state=42)
xgb = XGBClassifier(n_estimators=100, random_state=42, use_label_encoder=False)
svm = SVC(kernel='linear', probability=True, random_state=42)

# Create Voting Classifier
voting_model = VotingClassifier(estimators=[
    ('RandomForest', rf),
    ('XGBoost', xgb),
    ('SVM', svm)
], voting='soft')

# Train Voting Classifier
voting_model.fit(X_train, y_train)

# Make Predictions
y_pred_train = voting_model.predict(X_train)
y_pred_test = voting_model.predict(X_test)

# Evaluate Model
train_accuracy = accuracy_score(y_train, y_pred_train)
test_accuracy = accuracy_score(y_test, y_pred_test)

print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Testing Accuracy: {test_accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_test))

# SVM

In [ ]:
from sklearn.svm import SVC

# Train SVM Model
svm_model = SVC(kernel='linear', C=1.0, random_state=42)
svm_model.fit(X_train, y_train)

# Make Predictions
y_pred_train = svm_model.predict(X_train)
y_pred_test = svm_model.predict(X_test)

# Evaluate Model
train_accuracy = accuracy_score(y_train, y_pred_train)
test_accuracy = accuracy_score(y_test, y_pred_test)

print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Testing Accuracy: {test_accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_test))

# LSTM

# Tokenize and Pad the Data

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

# Tokenize the Text
max_words = 10000  # Maximum number of words in the vocabulary
max_length = 128   # Maximum sequence length



tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
# Fitting the tokenizer on the preprocessed text data from train_texts
tokenizer.fit_on_texts(train_texts)

# Convert text to sequence
train_sequences = tokenizer.texts_to_sequences(train_texts)
test_sequences = tokenizer.texts_to_sequences(test_texts)

# Pad sequences
X_train = pad_sequences(train_sequences, maxlen=max_length, padding='post', truncating='post')
X_test = pad_sequences(test_sequences, maxlen=max_length, padding='post', truncating='post')

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

# Build the LSTM Model

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.regularizers import L2

# Build the Enhanced LSTM Model
embedding_dim = 128

model = Sequential([

    Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=max_length),

    # Add a Bidirectional LSTM layer
    Bidirectional(LSTM(128, return_sequences=False, dropout=0.3, recurrent_dropout=0.3)),

    # Dense layer with L2 regularization
    Dense(128, activation='relu', kernel_regularizer=L2(0.01)),
    Dropout(0.5),

    # Add another Dense layer
    Dense(64, activation='relu', kernel_regularizer=L2(0.01)),
    Dropout(0.5),

    # Output layer for binary classification
    Dense(1, activation='sigmoid')
])

# Compile the model
model.compile(
    optimizer='adam',   # You can also try optimizers like RMSprop or SGD
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Model summary
model.summary()

# Train the LSTM Model

In [ ]:
# Train the model
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,              # Batch size
    validation_data=(X_test, y_test),
    verbose=2
)

In [ ]:

y_pred_prob = model.predict(X_test)

fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc="lower right")
plt.savefig('lstm_roc_curve.png') # Save the figure
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Plot training and validation accuracy
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

# Plot training and validation loss
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.show()

In [ ]:


from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, matthews_corrcoef


# Print evaluation results
accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds)
recall = recall_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)
mcc = matthews_corrcoef(all_labels, all_preds)

tn, fp, fn, tp = confusion_matrix(all_labels, all_preds).ravel()
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall (Sensitivity): {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Sensitivity: {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"Matthews Correlation Coefficient: {mcc:.4f}")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt


cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Reds",
            xticklabels=['Predicted 0', 'Predicted 1'],
            yticklabels=['Actual 0', 'Actual 1'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.savefig('Lstmconfusion_matrix.png')
plt.show()

# RoBERTa

In [ ]:
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from transformers import Trainer, TrainingArguments
from torch.utils.data import DataLoader, Dataset
import torch

# Tokenizer and Model
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
tokenizer.pad_token = tokenizer.eos_token
model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=2)


class BengaliDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(label, dtype=torch.long),
        }

# Prepare Data
train_dataset = BengaliDataset(train_texts, y_train, tokenizer, max_length=128)
test_dataset = BengaliDataset(test_texts, y_test, tokenizer, max_length=128)

# Training Arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    save_total_limit=2,
    logging_steps=10,
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

# Train the Model
trainer.train()

# Evaluate
trainer.evaluate()

In [ ]:

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, matthews_corrcoef, roc_auc_score
import numpy as np


predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

# Calculate metrics
accuracy = accuracy_score(labels, preds)
precision = precision_score(labels, preds)
recall = recall_score(labels, preds)
f1 = f1_score(labels, preds)
mcc = matthews_corrcoef(labels, preds)

tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)

try:
    roc_auc = roc_auc_score(labels, predictions.predictions[:, 1]) # Assuming binary classification
    print(f"ROC-AUC Score: {roc_auc:.4f}")
except ValueError:
    print("ROC-AUC Score could not be calculated. Check if labels are binary.")
    roc_auc = 0


print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall (Sensitivity): {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Sensitivity: {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"Matthews Correlation Coefficient: {mcc:.4f}")

# DistilBERT

In [ ]:
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification

# Tokenizer and Model
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
tokenizer.pad_token = tokenizer.eos_token
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)


# Tokenize and Train
train_dataset = BengaliDataset(train_texts, y_train, tokenizer, max_length=128)
test_dataset = BengaliDataset(test_texts, y_test, tokenizer, max_length=128)

# Use the same Trainer and TrainingArguments as shown in the RoBERTa example
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

# Train and Evaluate
trainer.train()
trainer.evaluate()

In [ ]:


from sklearn.metrics import roc_auc_score

predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

# Calculate metrics
accuracy = accuracy_score(labels, preds)
precision = precision_score(labels, preds)
recall = recall_score(labels, preds)
f1 = f1_score(labels, preds)
mcc = matthews_corrcoef(labels, preds)

tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)

try:
    roc_auc = roc_auc_score(labels, predictions.predictions[:, 1]) # Assuming binary classification
    print(f"ROC-AUC Score: {roc_auc:.4f}")
except ValueError:
    print("ROC-AUC Score could not be calculated. Check if labels are binary.")
    roc_auc = 0


print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall (Sensitivity): {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Sensitivity: {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"Matthews Correlation Coefficient: {mcc:.4f}")

# GPT-2

In [ ]:
from transformers import GPT2Tokenizer, GPT2ForSequenceClassification

# Tokenizer and Model
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
model = GPT2ForSequenceClassification.from_pretrained("gpt2", num_labels=2)

# Tokenize and Train
train_dataset = BengaliDataset(train_texts, y_train, tokenizer, max_length=128)
test_dataset = BengaliDataset(test_texts, y_test, tokenizer, max_length=128)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer.train()
trainer.evaluate()

In [ ]:

from sklearn.metrics import roc_auc_score

predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

# Calculate metrics
accuracy = accuracy_score(labels, preds)
precision = precision_score(labels, preds)
recall = recall_score(labels, preds)
f1 = f1_score(labels, preds)
mcc = matthews_corrcoef(labels, preds)

tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)

try:
    roc_auc = roc_auc_score(labels, predictions.predictions[:, 1]) # Assuming binary classification
    print(f"ROC-AUC Score: {roc_auc:.4f}")
except ValueError:
    print("ROC-AUC Score could not be calculated. Check if labels are binary.")
    roc_auc = 0

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall (Sensitivity): {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Sensitivity: {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"Matthews Correlation Coefficient: {mcc:.4f}")

# XLNet

In [ ]:
from transformers import XLNetTokenizer, XLNetForSequenceClassification

# Tokenizer and Model
tokenizer = XLNetTokenizer.from_pretrained("xlnet-base-cased")
tokenizer.pad_token = tokenizer.eos_token
model = XLNetForSequenceClassification.from_pretrained("xlnet-base-cased", num_labels=2)

# Tokenize and Train
train_dataset = BengaliDataset(train_texts, y_train, tokenizer, max_length=128)
test_dataset = BengaliDataset(test_texts, y_test, tokenizer, max_length=128)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer.train()
trainer.evaluate()

In [ ]:


from sklearn.metrics import roc_auc_score

predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

# Calculate metrics
accuracy = accuracy_score(labels, preds)
precision = precision_score(labels, preds)
recall = recall_score(labels, preds)
f1 = f1_score(labels, preds)
mcc = matthews_corrcoef(labels, preds)

tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)

try:
    roc_auc = roc_auc_score(labels, predictions.predictions[:, 1]) # Assuming binary classification
    print(f"ROC-AUC Score: {roc_auc:.4f}")
except ValueError:
    print("ROC-AUC Score could not be calculated. Check if labels are binary.")
    roc_auc = 0


print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall (Sensitivity): {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Sensitivity: {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"Matthews Correlation Coefficient: {mcc:.4f}")

# ALBERT

In [ ]:
from transformers import AlbertTokenizer, AlbertForSequenceClassification

# Tokenizer and Model
tokenizer = AlbertTokenizer.from_pretrained("albert-base-v2")
tokenizer.pad_token = tokenizer.eos_token
model = AlbertForSequenceClassification.from_pretrained("albert-base-v2", num_labels=2)

# Tokenize and Train
train_dataset = BengaliDataset(train_texts, y_train, tokenizer, max_length=128)
test_dataset = BengaliDataset(test_texts, y_test, tokenizer, max_length=128)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer.train()
trainer.evaluate()

In [ ]:


from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, matthews_corrcoef, roc_auc_score

# ... (Your existing code) ...

predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

# Calculate metrics
accuracy = accuracy_score(labels, preds)
precision = precision_score(labels, preds)
recall = recall_score(labels, preds)
f1 = f1_score(labels, preds)
mcc = matthews_corrcoef(labels, preds)

tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)

try:
    roc_auc = roc_auc_score(labels, predictions.predictions[:, 1])
    print(f"ROC-AUC Score: {roc_auc:.4f}")
except ValueError:
    print("ROC-AUC Score could not be calculated. Check if labels are binary.")
    roc_auc = 0

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall (Sensitivity): {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Sensitivity: {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"Matthews Correlation Coefficient: {mcc:.4f}")

# GRU

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout

embedding_dim = 128

model = Sequential([
    Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=max_length),
    GRU(128, return_sequences=False, dropout=0.3, recurrent_dropout=0.3),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')  # Binary classification output
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

# Train the GRU Model
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_test, y_test),
    verbose=2
)

# Evaluate
loss, accuracy = model.evaluate(X_test, y_test, verbose=2)
print(f"Test Accuracy: {accuracy:.4f}")

In [ ]:


from sklearn.metrics import roc_auc_score

y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int) # Convert probabilities to binary predictions


accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)

try:
    roc_auc = roc_auc_score(y_test, y_pred_prob)
    print(f"ROC-AUC Score: {roc_auc:.4f}")
except ValueError:
    print("ROC-AUC Score could not be calculated. Check if labels are binary.")
    roc_auc = 0

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall (Sensitivity): {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Sensitivity: {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"Matthews Correlation Coefficient: {mcc:.4f}")

Transformer Encoder 1 (Custom Transformer)

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Embedding, Dropout, LayerNormalization, MultiHeadAttention

def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
    x = MultiHeadAttention(key_dim=head_size, num_heads=num_heads)(inputs, inputs)
    x = Dropout(dropout)(x)
    x = LayerNormalization(epsilon=1e-6)(x)
    res = x + inputs

    x = Dense(ff_dim, activation="relu")(res)
    x = Dropout(dropout)(x)
    x = Dense(inputs.shape[-1])(x)
    x = LayerNormalization(epsilon=1e-6)(x)
    return x + res

inputs = Input(shape=(max_length,))
embedding_layer = Embedding(input_dim=max_words, output_dim=128)(inputs)
x = transformer_encoder(embedding_layer, head_size=128, num_heads=4, ff_dim=128, dropout=0.3)
x = Dense(64, activation="relu")(x)
x = Dropout(0.5)(x)
outputs = Dense(1, activation="sigmoid")(x)

model = Model(inputs, outputs)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

# Train the Transformer Encoder Model
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_test, y_test),
    verbose=2
)

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, matthews_corrcoef
)

def evaluate_model(y_true, y_pred, y_pred_proba=None):

    # Accuracy
    accuracy = accuracy_score(y_true, y_pred)

    # Precision
    precision = precision_score(y_true, y_pred)

    # Recall (Sensitivity)
    recall = recall_score(y_true, y_pred)

    # F1 Score
    f1 = f1_score(y_true, y_pred)

    # Confusion Matrix
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    # Specificity
    specificity = tn / (tn + fp)

    # ROC-AUC Score
    roc_auc = roc_auc_score(y_true, y_pred_proba) if y_pred_proba is not None else None

    # Matthews Correlation Coefficient (MCC)
    mcc = matthews_corrcoef(y_true, y_pred)

    # Print all metrics
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall (Sensitivity): {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"Specificity: {specificity:.4f}")
    if roc_auc is not None:
        print(f"ROC-AUC Score: {roc_auc:.4f}")
    print(f"Matthews Correlation Coefficient (MCC): {mcc:.4f}")

    # Return all metrics as a dictionary
    return {
        "Accuracy": round(accuracy, 4),
        "Precision": round(precision, 4),
        "Recall (Sensitivity)": round(recall, 4),
        "F1 Score": round(f1, 4),
        "Specificity": round(specificity, 4),
        "ROC-AUC Score": round(roc_auc, 4) if roc_auc is not None else None,
        "MCC": round(mcc, 4)
    }


metrics = evaluate_model(y_test, y_pred, y_pred_proba=y_pred_proba)

# Custom CNN for Text Classification

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout

# Define the CNN Model
embedding_dim = 128

model = Sequential([
    Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=max_length),
    Conv1D(filters=128, kernel_size=5, activation='relu'),  # Convolution layer
    GlobalMaxPooling1D(),  # Global Pooling
    Dense(128, activation='relu'),  # Fully connected layer
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')  # Output layer for binary classification
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

# Train the CNN Model
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_test, y_test),
    verbose=2
)

# Evaluate the Model
loss, accuracy = model.evaluate(X_test, y_test, verbose=2)
print(f"Test Accuracy: {accuracy:.4f}")

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, matthews_corrcoef
)

def evaluate_model(y_true, y_pred, y_pred_proba=None):

    # Accuracy
    accuracy = accuracy_score(y_true, y_pred)

    # Precision
    precision = precision_score(y_true, y_pred)

    # Recall (Sensitivity)
    recall = recall_score(y_true, y_pred)

    # F1 Score
    f1 = f1_score(y_true, y_pred)

    # Confusion Matrix
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    # Specificity
    specificity = tn / (tn + fp)

    # ROC-AUC Score
    roc_auc = roc_auc_score(y_true, y_pred_proba) if y_pred_proba is not None else None

    # Matthews Correlation Coefficient (MCC)
    mcc = matthews_corrcoef(y_true, y_pred)

    # Print all metrics
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall (Sensitivity): {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"Specificity: {specificity:.4f}")
    if roc_auc is not None:
        print(f"ROC-AUC Score: {roc_auc:.4f}")
    print(f"Matthews Correlation Coefficient (MCC): {mcc:.4f}")

    # Return all metrics as a dictionary
    return {
        "Accuracy": round(accuracy, 4),
        "Precision": round(precision, 4),
        "Recall (Sensitivity)": round(recall, 4),
        "F1 Score": round(f1, 4),
        "Specificity": round(specificity, 4),
        "ROC-AUC Score": round(roc_auc, 4) if roc_auc is not None else None,
        "MCC": round(mcc, 4)
    }


metrics = evaluate_model(y_test, y_pred, y_pred_proba=y_pred_proba)

Embedding Layer: Converts words into dense vectors of fixed size.
E
=
Embedding
(
X
)
;
E
∈
R
n
×
d
E=Embedding(X);E∈R
n×d

Where
n
n is the sequence length, and
d
d is the embedding dimension.
Convolution Operation: Applies a convolutional filter
W
W over the input embeddings:
C
i
=
ReLU
(
W
⋅
E
i
:
i
+
k
−
1
+
b
)
C
i
​
 =ReLU(W⋅E
i:i+k−1
​
 +b)
Where
k
k is the kernel size,
b
b is the bias, and
ReLU
ReLU is the activation function.
Global Max Pooling: Selects the maximum value from each feature map:
P
=
max
⁡
(
C
)
P=max(C)
Dense Layers: Fully connected layers with dropout regularization:
h
=
ReLU
(
W
d
⋅
P
+
b
d
)
h=ReLU(W
d
​
 ⋅P+b
d
​
 )
Output Layer: Produces a probability score for binary classification:
y
^
=
σ
(
W
o
⋅
h
+
b
o
)
y
^
​
 =σ(W
o
​
 ⋅h+b
o
​
 )
Where
σ
σ is the sigmoid activation.

# Hybrid LSTM-CNN Model

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Conv1D, GlobalMaxPooling1D, Dense, Dropout

# Define the LSTM-CNN Model
embedding_dim = 128

model = Sequential([
    Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=max_length),
    LSTM(128, return_sequences=True, dropout=0.3, recurrent_dropout=0.3),  # LSTM Layer
    Conv1D(filters=64, kernel_size=3, activation='relu'),  # Convolution Layer
    GlobalMaxPooling1D(),  # Global Pooling
    Dense(128, activation='relu'),  # Fully connected layer
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')  # Output layer for binary classification
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

# Train the Hybrid Model
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_test, y_test),
    verbose=2
)

# Evaluate the Model
loss, accuracy = model.evaluate(X_test, y_test, verbose=2)
print(f"Test Accuracy: {accuracy:.4f}")

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, matthews_corrcoef
)

def evaluate_model(y_true, y_pred, y_pred_proba=None):

    # Accuracy
    accuracy = accuracy_score(y_true, y_pred)

    # Precision
    precision = precision_score(y_true, y_pred)

    # Recall (Sensitivity)
    recall = recall_score(y_true, y_pred)

    # F1 Score
    f1 = f1_score(y_true, y_pred)

    # Confusion Matrix
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    # Specificity
    specificity = tn / (tn + fp)

    # ROC-AUC Score
    roc_auc = roc_auc_score(y_true, y_pred_proba) if y_pred_proba is not None else None

    # Matthews Correlation Coefficient (MCC)
    mcc = matthews_corrcoef(y_true, y_pred)

    # Print all metrics
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall (Sensitivity): {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"Specificity: {specificity:.4f}")
    if roc_auc is not None:
        print(f"ROC-AUC Score: {roc_auc:.4f}")
    print(f"Matthews Correlation Coefficient (MCC): {mcc:.4f}")

    # Return all metrics as a dictionary
    return {
        "Accuracy": round(accuracy, 4),
        "Precision": round(precision, 4),
        "Recall (Sensitivity)": round(recall, 4),
        "F1 Score": round(f1, 4),
        "Specificity": round(specificity, 4),
        "ROC-AUC Score": round(roc_auc, 4) if roc_auc is not None else None,
        "MCC": round(mcc, 4)
    }


metrics = evaluate_model(y_test, y_pred, y_pred_proba=y_pred_proba)

Embedding Layer: Same as the CNN model:
E
=
Embedding
(
X
)
;
E
∈
R
n
×
d
E=Embedding(X);E∈R
n×d

LSTM Layer: Processes the sequence to capture temporal dependencies:
h
t
,
c
t
=
LSTM
(
E
t
,
h
t
−
1
,
c
t
−
1
)
h
t
​
 ,c
t
​
 =LSTM(E
t
​
 ,h
t−1
​
 ,c
t−1
​
 )
Here,
h
t
h
t
​
  is the hidden state and
c
t
c
t
​
  is the cell state at time step
t
t.
For the entire sequence:
H
=
[
h
1
,
h
2
,
…
,
h
n
]
;
H
∈
R
n
×
d
h
H=[h
1
​
 ,h
2
​
 ,…,h
n
​
 ];H∈R
n×d
h
​


Where
d
h
d
h
​
  is the LSTM's hidden size.
Convolution Operation: Applies a convolutional filter
W
W over the LSTM outputs:
C
i
=
ReLU
(
W
⋅
H
i
:
i
+
k
−
1
+
b
)
C
i
​
 =ReLU(W⋅H
i:i+k−1
​
 +b)
Global Max Pooling: Selects the maximum value from each feature map:
P
=
max
⁡
(
C
)
P=max(C)
Dense Layers: Same as the CNN model:
h
=
ReLU
(
W
d
⋅
P
+
b
d
)
h=ReLU(W
d
​
 ⋅P+b
d
​
 )
Output Layer: Same as the CNN model:
y
^
=
σ
(
W
o
⋅
h
+
b
o
)
y
^
​
 =σ(W
o
​
 ⋅h+b
o
​
 )

Custom CNN Model:
Best for tasks where local relationships (e.g., n-grams) are important.
LaTeX Operations: Involves embedding, convolution, max pooling, and dense layers.
Hybrid LSTM-CNN Model:
Best for tasks where both sequential dependencies and local patterns matter.
LaTeX Operations: Combines LSTM hidden states with convolution and pooling.


# Advanced Hybrid Model: Transformer + BiLSTM + CNN

Transformer Encoder: Captures long-range dependencies in the text using self-attention.
BiLSTM (Bidirectional LSTM): Captures sequential dependencies in both forward and backward directions.
CNN: Extracts local spatial features (like n-grams) from the sequential output of BiLSTM.
Attention Mechanism: Adds a custom attention layer to focus on important parts of the sequence.
Dense Layers: Adds fully connected layers for classification.

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Embedding, Bidirectional, LSTM, Conv1D, GlobalMaxPooling1D, Dense,
    Dropout, LayerNormalization, MultiHeadAttention, Add, Flatten
)

# Define the Advanced Hybrid Model
def Transformer_LSTM_CNN(max_words, max_length, embedding_dim=128, num_heads=4, ff_dim=128, num_classes=1):
    # Input Layer
    inputs = Input(shape=(max_length,), dtype="int32")

    # Embedding Layer
    embedding_layer = Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=max_length)(inputs)

    # Transformer Encoder Layer
    attention_output = MultiHeadAttention(num_heads=num_heads, key_dim=embedding_dim)(embedding_layer, embedding_layer)
    attention_output = Dropout(0.3)(attention_output)
    attention_output = Add()([embedding_layer, attention_output])  # Residual connection
    attention_output = LayerNormalization(epsilon=1e-6)(attention_output)

    # Feedforward Layer in Transformer Encoder
    ff_output = Dense(ff_dim, activation="relu")(attention_output)
    ff_output = Dense(embedding_dim)(ff_output)
    ff_output = Add()([attention_output, ff_output])  # Residual connection
    transformer_output = LayerNormalization(epsilon=1e-6)(ff_output)

    # BiLSTM Layer
    lstm_output = Bidirectional(LSTM(128, return_sequences=True, dropout=0.3, recurrent_dropout=0.3))(transformer_output)

    # Convolution Layer (1D CNN)
    cnn_output = Conv1D(filters=64, kernel_size=3, activation="relu")(lstm_output)
    global_pooling = GlobalMaxPooling1D()(cnn_output)

    # Fully Connected Layers with Dropout
    dense_output = Dense(128, activation="relu")(global_pooling)
    dense_output = Dropout(0.5)(dense_output)
    dense_output = Dense(64, activation="relu")(dense_output)
    dense_output = Dropout(0.5)(dense_output)

    # Output Layer
    outputs = Dense(num_classes, activation="sigmoid")(dense_output)  # Sigmoid for binary classification

    # Create Model
    model = Model(inputs, outputs)
    return model

# Instantiate and Compile the Model
model = Transformer_LSTM_CNN(max_words=max_words, max_length=max_length)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

# Train the Model
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=10,
    batch_size=32,
    verbose=2
)

# Evaluate the Model
loss, accuracy = model.evaluate(X_test, y_test, verbose=2)
print(f"Test Accuracy: {accuracy:.4f}")

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, matthews_corrcoef
)

def evaluate_model(y_true, y_pred, y_pred_proba=None):

    # Accuracy
    accuracy = accuracy_score(y_true, y_pred)

    # Precision
    precision = precision_score(y_true, y_pred)

    # Recall (Sensitivity)
    recall = recall_score(y_true, y_pred)

    # F1 Score
    f1 = f1_score(y_true, y_pred)

    # Confusion Matrix
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    # Specificity
    specificity = tn / (tn + fp)

    # ROC-AUC Score
    roc_auc = roc_auc_score(y_true, y_pred_proba) if y_pred_proba is not None else None

    # Matthews Correlation Coefficient (MCC)
    mcc = matthews_corrcoef(y_true, y_pred)

    # Print all metrics
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall (Sensitivity): {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"Specificity: {specificity:.4f}")
    if roc_auc is not None:
        print(f"ROC-AUC Score: {roc_auc:.4f}")
    print(f"Matthews Correlation Coefficient (MCC): {mcc:.4f}")

    # Return all metrics as a dictionary
    return {
        "Accuracy": round(accuracy, 4),
        "Precision": round(precision, 4),
        "Recall (Sensitivity)": round(recall, 4),
        "F1 Score": round(f1, 4),
        "Specificity": round(specificity, 4),
        "ROC-AUC Score": round(roc_auc, 4) if roc_auc is not None else None,
        "MCC": round(mcc, 4)
    }


metrics = evaluate_model(y_test, y_pred, y_pred_proba=y_pred_proba)

Explanation of the Architecture
Embedding Layer:
Converts word indices into dense vectors of fixed size.
E
=
Embedding
(
X
)
;
E
∈
R
n
×
d
E=Embedding(X);E∈R
n×d

Where
n
n is the sequence length, and
d
d is the embedding dimension.
Transformer Encoder:
The Multi-Head Attention mechanism captures long-range dependencies in the text.
Attention output:
Attention
(
Q
,
K
,
V
)
=
Softmax
(
Q
K
T
d
k
)
V
Attention(Q,K,V)=Softmax(
d
k
​

​

QK
T

​
 )V
Here:
Q
,
K
,
V
Q,K,V are the query, key, and value matrices derived from the input embeddings.
d
k
d
k
​
  is the dimension of the key.
The attention output is passed through a feedforward network with residual connections and layer normalization.
BiLSTM Layer:
Processes the transformer output in both forward and backward directions to capture sequential dependencies.
Forward LSTM:
h
t
→
=
LSTM
(
E
t
,
h
t
−
1
→
)
h
t
​

​
 =LSTM(E
t
​
 ,
h
t−1
​

​
 )
Backward LSTM:
h
t
←
=
LSTM
(
E
t
,
h
t
+
1
←
)
h
t
​

​
 =LSTM(E
t
​
 ,
h
t+1
​

​
 )
BiLSTM output:
H
t
=
[
h
t
→
,
h
t
←
]
H
t
​
 =[
h
t
​

​
 ,
h
t
​

​
 ]
Convolution Layer:
Applies a 1D convolution over the LSTM outputs to capture local spatial patterns (like n-grams).
C
i
=
ReLU
(
W
⋅
H
i
:
i
+
k
−
1
+
b
)
C
i
​
 =ReLU(W⋅H
i:i+k−1
​
 +b)
Global Max Pooling:
Selects the most important feature from each feature map.
P
=
max
⁡
(
C
)
P=max(C)
Dense Layers:
Fully connected layers with dropout regularization to reduce overfitting.
h
=
ReLU
(
W
d
⋅
P
+
b
d
)
h=ReLU(W
d
​
 ⋅P+b
d
​
 )
Output Layer:
Produces a probability score for binary classification.
y
^
=
σ
(
W
o
⋅
h
+
b
o
)
y
^
​
 =σ(W
o
​
 ⋅h+b
o
​
 )

Transformer Encoder:
Attention:
Attention
(
Q
,
K
,
V
)
=
Softmax
(
Q
K
T
d
k
)
V
Attention(Q,K,V)=Softmax(
d
k
​

​

QK
T

​
 )V
Residual Connection:
Z
=
LayerNorm
(
X
+
Attention
(
Q
,
K
,
V
)
)
Z=LayerNorm(X+Attention(Q,K,V))
BiLSTM:
Forward and Backward Hidden States:
H
t
=
[
h
t
→
,
h
t
←
]
H
t
​
 =[
h
t
​

​
 ,
h
t
​

​
 ]
Convolution:
Local Feature Extraction:
C
i
=
ReLU
(
W
⋅
H
i
:
i
+
k
−
1
+
b
)
C
i
​
 =ReLU(W⋅H
i:i+k−1
​
 +b)
Pooling:
Global Max Pooling:
P
=
max
⁡
(
C
)
P=max(C)
Fully Connected Layers:
Dense Layer:
h
=
ReLU
(
W
d
⋅
P
+
b
d
)
h=ReLU(W
d
​
 ⋅P+b
d
​
 )
Output:
Sigmoid Activation for Binary Classification:
y
^
=
σ
(
W
o
⋅
h
+
b
o
)
y
^
​
 =σ(W
o
​
 ⋅h+b
o
​
 )
Why This Model is Creative and Powerful
Transformer Encoder: Captures long-range dependencies effectively.
BiLSTM: Models sequential relationships in both directions.
CNN: Extracts local n-gram level features.
Combining Architectures: This hybrid approach leverages the best of transformers, RNNs, and CNNs.
This model should perform better than standalone architectures like CNN, LSTM, or standard transformers.
